# Prepare First-Pass SCEPTER Inputs

This notebook starts the SCEPTER part of the ERW MRV workflow. It uses the ESA WorldCover cropland raster as the spatial model-unit source.

At this stage we are not yet running SCEPTER. We are preparing clean, auditable input tables: one table for raster-derived model units, one table for ERW scenarios, and one expanded run table with every model-unit/scenario combination.


## Workflow

This notebook stages test-ready SCEPTER inputs from the prepared spatial layers:

1. Cropland map from the ESA WorldCover extraction (`erw_cdr_ugagric`).
2. HWSD2 AOI soil defaults and soil-type polygons from notebook 03b.
3. CHIRPS monthly rainfall summary from the climate-prep step.
4. ERW scenarios expanded across the prepared model units.

The notebook still writes only the staging inputs under `data/scepter_runs/inputs`. Notebook 05 remains responsible for creating per-run SCEPTER configuration folders and execution manifests.


In [1]:
from pathlib import Path
import os
import sys

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.features import shapes
from rasterio.mask import mask
from rasterio.windows import Window
from shapely.geometry import box, mapping, shape
from shapely.ops import unary_union

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import SCEPTER_INPUTS, ensure_dir
from erw_mrv.scepter import (
    DEFAULT_SCENARIOS,
    ScepterDefaults,
    expand_scepter_runs,
    make_model_units,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


PROJECT_ROOT = /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv
SOURCE_PROJECT_ROOT = /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv
DATA_ROOT = /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data


## Configure Inputs

The spatial source is the ESA cropland raster exported by the landcover notebook. The notebook builds coarse SCEPTER model units from cropland raster blocks, then attaches HWSD2 soil-map labels and CHIRPS rainfall defaults.


In [2]:
OUTPUT_TAG = "chirps_2021_test"
SOIL_DIR = DATA_ROOT / "processed" / "soil" / "hwsd2"
SOIL_DEFAULTS_PATH = SOIL_DIR / "scepter_soil_defaults_hwsd2.csv"
SOIL_TYPES_PATH = SOIL_DIR / "hwsd2_aoi_soil_types.geojson"
RAINFALL_SUMMARY_PATH = DATA_ROOT / "processed" / "climate" / "rainfall" / "monthly_rainfall_aoi_jan_jun_2026.csv"

WORKSPACE_ROOT = SOURCE_PROJECT_ROOT.parent
UGAGRIC_DATA_ROOT = WORKSPACE_ROOT / "erw_cdr_ugagric" / "data"
CROPLAND_RASTER_CANDIDATES = [
    DATA_ROOT / "processed" / "landuse_landcover" / "ug_agric_21landuse.tif",
    UGAGRIC_DATA_ROOT / "processed" / "landuse_landcover" / "ug_agric_21landuse.tif",
]
CROPLAND_RASTER_PATH = next((path for path in CROPLAND_RASTER_CANDIDATES if path.exists()), CROPLAND_RASTER_CANDIDATES[0])

REQUIRE_REAL_SOIL = True
REQUIRE_REAL_RAINFALL = True
RUNOFF_FRACTION_OF_PRECIPITATION = 0.25

# Raster settings. Each output unit is a block containing cropland pixels.
RASTER_BLOCK_SIZE = 512
MIN_CROPLAND_PIXELS_PER_UNIT = 25
NOMINAL_GEOGRAPHIC_CROPLAND_PIXEL_AREA_M2 = 100.0  # ESA WorldCover is a 10 m product.
INPUT_MODE = "esa_cropland_raster_blocks"

if not CROPLAND_RASTER_PATH.exists():
    raise FileNotFoundError(
        "ESA cropland raster not found. Run the landcover notebook first. Checked: "
        f"{CROPLAND_RASTER_PATH}. Checked candidates: {CROPLAND_RASTER_CANDIDATES}"
    )

SCEPTER_INPUT_DIR = ensure_dir(SCEPTER_INPUTS)
print(f"Using ESA cropland raster blocks: {CROPLAND_RASTER_PATH}")
print(f"SCEPTER input directory for notebook 04: {SCEPTER_INPUT_DIR}")
print("Notebook 04 writes only data/scepter_runs/inputs. Notebook 05 creates data/scepter_runs/runs and data/scepter_runs/outputs.")
SCEPTER_INPUT_DIR


Using ESA cropland raster blocks: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_agric_21landuse.tif
SCEPTER input directory for notebook 04: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs
Notebook 04 writes only data/scepter_runs/inputs. Notebook 05 creates data/scepter_runs/runs and data/scepter_runs/outputs.


PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs')

## Load Raster Model Units

This cell creates coarse spatial units from blocks of the ESA cropland raster. `MAX_TEST_UNITS` keeps notebook 04 lean while testing; set it to `None` when you want every available cropland block staged.


In [3]:
def optional_positive_int_from_env(name: str) -> int | None:
    raw = os.environ.get(name, "").strip()
    if raw == "" or raw.lower() in {"all", "none", "null"}:
        return None
    value = int(raw)
    if value <= 0:
        raise ValueError(f"{name} must be a positive integer, or unset/'all' to stage every cropland unit. Got: {raw}")
    return value


MAX_MODEL_UNITS = optional_positive_int_from_env("ERW_MAX_MODEL_UNITS")
# Default is all ESA cropland blocks. Set ERW_MAX_MODEL_UNITS=100, 500, etc. for a quicker test build.



def cropland_pixel_area_m2(src) -> tuple[float, str]:
    if src.crs and src.crs.is_geographic:
        return (
            NOMINAL_GEOGRAPHIC_CROPLAND_PIXEL_AREA_M2,
            "geographic raster; used ESA WorldCover nominal 10 m pixel area",
        )
    return abs(src.transform.a * src.transform.e), "projected raster transform area"


def cropland_geometry_from_window(data: np.ndarray, transform):
    """Return a geometry following cropland pixels inside one raster window."""
    cropland_mask = data == 1
    if not cropland_mask.any():
        return None
    geometries = [
        shape(geometry)
        for geometry, value in shapes(
            cropland_mask.astype("uint8"),
            mask=cropland_mask,
            transform=transform,
        )
        if value == 1
    ]
    if not geometries:
        return None
    geometry = unary_union(geometries)
    if geometry.is_empty:
        return None
    return geometry


def load_cropland_raster_blocks(
    raster_path: Path,
    block_size: int = RASTER_BLOCK_SIZE,
    min_pixels: int = MIN_CROPLAND_PIXELS_PER_UNIT,
) -> gpd.GeoDataFrame:
    records = []
    with rasterio.open(raster_path) as src:
        pixel_area_m2, area_note = cropland_pixel_area_m2(src)
        for row_start in range(0, src.height, block_size):
            height = min(block_size, src.height - row_start)
            for col_start in range(0, src.width, block_size):
                width = min(block_size, src.width - col_start)
                window = Window(col_start, row_start, width, height)
                data = src.read(1, window=window)
                cropland_pixels = int((data == 1).sum())
                if cropland_pixels < min_pixels:
                    continue

                left, bottom, right, top = src.window_bounds(window)
                cropland_geometry = cropland_geometry_from_window(data, src.window_transform(window))
                if cropland_geometry is None:
                    continue

                cropland_area_m2 = cropland_pixels * pixel_area_m2
                records.append(
                    {
                        "value": 1,
                        "source": "cropland_raster_block",
                        "cropland_block_id": f"block_r{row_start:05d}_c{col_start:05d}",
                        "cropland_pixels": cropland_pixels,
                        "cropland_pixel_area_m2": pixel_area_m2,
                        "cropland_area_m2": cropland_area_m2,
                        "cropland_area_ha": cropland_area_m2 / 10_000,
                        "area_m2": cropland_area_m2,
                        "area_ha": cropland_area_m2 / 10_000,
                        "cropland_source_path": str(raster_path),
                        "cropland_area_note": area_note,
                        "cropland_geometry_note": "geometry follows ESA cropland pixels within raster block",
                        "block_bounds_geometry": box(left, bottom, right, top).wkt,
                        "geometry": cropland_geometry,
                    }
                )

        crs = src.crs

    if not records:
        raise ValueError(f"No cropland raster blocks found in {raster_path}")
    blocks = gpd.GeoDataFrame(records, crs=crs)
    return blocks.sort_values(["area_ha", "cropland_block_id"], ascending=[False, True]).reset_index(drop=True)


all_model_unit_blocks = load_cropland_raster_blocks(CROPLAND_RASTER_PATH)
model_unit_blocks = all_model_unit_blocks.copy()
if MAX_MODEL_UNITS is not None:
    model_unit_blocks = model_unit_blocks.head(MAX_MODEL_UNITS).copy()

print(f"Input mode: {INPUT_MODE}")
print(f"Available ESA cropland blocks: {len(all_model_unit_blocks):,}")
print(f"Loaded raster model units: {len(model_unit_blocks):,}")
print(f"Total cropland area represented: {model_unit_blocks['area_ha'].sum():,.2f} ha")
if MAX_MODEL_UNITS is not None:
    missing_area = all_model_unit_blocks['area_ha'].sum() - model_unit_blocks['area_ha'].sum()
    print(f"WARNING: ERW_MAX_MODEL_UNITS={MAX_MODEL_UNITS:,} excludes {missing_area:,.2f} ha of cropland from SCEPTER staging.")
else:
    print("ERW_MAX_MODEL_UNITS is unset/all; staging every ESA cropland block.")
model_unit_blocks[["area_ha"]].describe()


Input mode: esa_cropland_raster_blocks
Loaded raster model units: 100
Total cropland area represented: 90,763.12 ha


,area_ha
count,100.000000
mean,907.631200
std,171.394031
min,677.080000
25%,770.970000
50%,878.205000
75%,998.292500
max,1386.310000


## Create Model Units

These units include geometry, area, centroid coordinates, HWSD2 soil defaults, and CHIRPS rainfall. Runoff is still an explicit first-pass estimate derived from rainfall until a runoff layer is added.


In [4]:
def soil_texture_label(row):
    clay = pd.to_numeric(row.get("CLAY"), errors="coerce")
    sand = pd.to_numeric(row.get("SAND"), errors="coerce")
    silt = pd.to_numeric(row.get("SILT"), errors="coerce")
    wrb = str(row.get("WRB4", "")).strip().upper()

    if wrb == "WR" or pd.isna(clay) or clay < 0:
        return "Water / no soil"
    if clay >= 45:
        return "Clay"
    if clay >= 35:
        return "Clay loam"
    if sand >= 70:
        return "Sandy soil"
    if sand >= 50 and clay < 20:
        return "Sandy loam"
    if silt >= 50 and clay < 27:
        return "Silty loam"
    if clay >= 27:
        return "Clay loam"
    if clay >= 18:
        return "Loam"
    return "Light loam"


def numeric_or_default(value, default):
    value = pd.to_numeric(value, errors="coerce")
    return default if pd.isna(value) else float(value)


def resolve_existing_path(path_value, fallback_dir: Path | None = None) -> Path:
    path = Path(str(path_value))
    if path.exists():
        return path
    if fallback_dir is not None:
        fallback = fallback_dir / path.name
        if fallback.exists():
            return fallback
    return path


def load_soil_defaults(path: Path, require_real: bool = REQUIRE_REAL_SOIL) -> tuple[dict, dict]:
    if not path.exists():
        message = f"HWSD2 soil defaults not found: {path}. Run notebook 03b first."
        if require_real:
            raise FileNotFoundError(message)
        print(f"Warning: {message} Using explicit placeholder soil values.")
        return {}, {"soil_source": "placeholder_defaults", "soil_source_path": str(path)}

    soil = pd.read_csv(path).iloc[0].to_dict()
    usable = {}
    for key in ["soil_ph", "cec_cmolc_kg", "clay_pct", "bulk_density_g_cm3", "soil_depth_cm"]:
        value = pd.to_numeric(soil.get(key), errors="coerce")
        if pd.notna(value):
            usable[key] = float(value)
    metadata = {
        "soil_source": str(soil.get("source", "hwsd2")),
        "soil_source_path": str(path),
        "soil_note": str(soil.get("note", "AOI-weighted HWSD2 defaults")),
    }
    print(f"Using HWSD2 soil defaults from: {path}")
    return usable, metadata


def load_soil_types(path: Path) -> gpd.GeoDataFrame:
    if not path.exists():
        raise FileNotFoundError(f"HWSD2 AOI soil type polygons not found: {path}. Run notebook 03b first.")
    soil_types = gpd.read_file(path)
    if soil_types.empty:
        raise ValueError(f"HWSD2 AOI soil type polygons are empty: {path}")
    soil_types["soil_texture_group"] = soil_types.apply(soil_texture_label, axis=1)
    return soil_types


def attach_soil_map_attributes(units: gpd.GeoDataFrame, soil_types: gpd.GeoDataFrame, soil_path: Path) -> gpd.GeoDataFrame:
    units_with_soil = units.copy()
    soil_columns = [
        column
        for column in [
            "hwsd2_unit_id",
            "WRB4",
            "FAO90",
            "TEXTURE_USDA",
            "TEXTURE_SOTER",
            "CLAY",
            "SAND",
            "SILT",
            "PH_WATER",
            "CEC_SOIL",
            "CEC_CLAY",
            "CEC_EFF",
            "BULK",
            "REF_BULK",
            "soil_texture_group",
        ]
        if column in soil_types.columns
    ]
    soil_lookup = soil_types[soil_columns + ["geometry"]].to_crs(units_with_soil.crs)

    centroids = units_with_soil[["model_unit_id", "geometry"]].copy()
    centroids["geometry"] = centroids.geometry.representative_point()
    joined = gpd.sjoin(centroids, soil_lookup, how="left", predicate="within")
    joined = joined.drop(columns=[column for column in ["index_right"] if column in joined.columns])
    joined = joined.drop_duplicates(subset="model_unit_id")

    rename_map = {
        "hwsd2_unit_id": "soil_map_hwsd2_unit_id",
        "WRB4": "soil_map_wrb4",
        "FAO90": "soil_map_fao90",
        "TEXTURE_USDA": "soil_map_texture_usda",
        "TEXTURE_SOTER": "soil_map_texture_soter",
        "CLAY": "soil_map_clay_pct",
        "SAND": "soil_map_sand_pct",
        "SILT": "soil_map_silt_pct",
        "PH_WATER": "soil_map_ph",
        "CEC_SOIL": "soil_map_cec_cmolc_kg",
        "CEC_CLAY": "soil_map_cec_clay_cmolc_kg",
        "CEC_EFF": "soil_map_cec_eff_cmolc_kg",
        "BULK": "soil_map_bulk_density_g_cm3",
        "REF_BULK": "soil_map_ref_bulk_density_g_cm3",
        "soil_texture_group": "soil_map_texture_group",
    }
    joined = joined[["model_unit_id"] + [column for column in soil_columns if column in joined.columns]].rename(columns=rename_map)
    units_with_soil = units_with_soil.merge(joined, on="model_unit_id", how="left")
    units_with_soil["soil_map_source_path"] = str(soil_path)
    units_with_soil["soil_map_join"] = "representative_point_within_hwsd2_aoi_soil_type"

    soil_map_missing = units_with_soil["soil_map_texture_group"].isna()
    if soil_map_missing.any():
        missing_count = int(soil_map_missing.sum())
        print(
            f"Warning: {missing_count:,} model units did not intersect a HWSD2 soil polygon at their "
            "representative point; using AOI/default soil inputs for those units."
        )
        units_with_soil.loc[soil_map_missing, "soil_map_texture_group"] = "Unmapped soil - using AOI defaults"
        units_with_soil.loc[soil_map_missing, "soil_map_join"] = "no_hwsd2_representative_point_match_using_aoi_defaults"
    return units_with_soil


def apply_per_unit_soil_inputs(units: gpd.GeoDataFrame, defaults: dict) -> gpd.GeoDataFrame:
    units = units.copy()
    field_map = {
        "soil_ph": "soil_map_ph",
        "cec_cmolc_kg": "soil_map_cec_cmolc_kg",
        "clay_pct": "soil_map_clay_pct",
        "bulk_density_g_cm3": "soil_map_bulk_density_g_cm3",
    }
    for target, source in field_map.items():
        fallback = defaults[target]
        if source in units.columns:
            values = pd.to_numeric(units[source], errors="coerce")
            units[target] = values.where(values.notna(), fallback).astype(float)
        else:
            units[target] = fallback
    units["soil_input_note"] = "SCEPTER soil fields use per-model-unit HWSD2 representative-point attributes where available; otherwise AOI/default values."
    return units


def load_rainfall_defaults(path: Path, require_real: bool = REQUIRE_REAL_RAINFALL) -> tuple[dict, dict, pd.DataFrame]:
    if not path.exists():
        message = f"CHIRPS rainfall summary not found: {path}. Run notebook 03a first."
        if require_real:
            raise FileNotFoundError(message)
        print(f"Warning: {message} Using placeholder precipitation.")
        precipitation_mm_yr = 1200.0
        fallback = pd.DataFrame()
        return (
            {"precipitation_mm_yr": precipitation_mm_yr, "runoff_mm_yr": precipitation_mm_yr * RUNOFF_FRACTION_OF_PRECIPITATION},
            {"rainfall_source": "placeholder_defaults", "rainfall_source_path": str(path), "rainfall_months_used": ""},
            fallback,
        )

    rainfall = pd.read_csv(path)
    if rainfall.empty or "rainfall_mm" not in rainfall.columns:
        raise ValueError(f"Rainfall summary is empty or missing rainfall_mm: {path}")

    rainfall_values = pd.to_numeric(rainfall["rainfall_mm"], errors="coerce").dropna()
    if rainfall_values.empty:
        raise ValueError(f"Rainfall summary has no numeric rainfall_mm values: {path}")

    months_used = rainfall["month"].astype(str).tolist() if "month" in rainfall.columns else []
    precipitation_mm_yr = float(rainfall_values.sum() * (12 / len(rainfall_values)))
    runoff_mm_yr = float(precipitation_mm_yr * RUNOFF_FRACTION_OF_PRECIPITATION)
    metadata = {
        "rainfall_source": "CHIRPS rainfall_chirps_monthly via DE Africa S3",
        "rainfall_source_path": str(path),
        "rainfall_months_used": ",".join(months_used),
        "missing_requested_months": str(rainfall.get("missing_requested_months", pd.Series([""])).iloc[0]),
        "runoff_note": f"runoff_mm_yr estimated as {RUNOFF_FRACTION_OF_PRECIPITATION:.2f} * precipitation_mm_yr",
    }
    print(f"Using CHIRPS rainfall summary from: {path}")
    print(f"AOI annualized precipitation fallback: {precipitation_mm_yr:.1f} mm/yr")
    print(f"AOI estimated runoff fallback: {runoff_mm_yr:.1f} mm/yr")
    return {"precipitation_mm_yr": precipitation_mm_yr, "runoff_mm_yr": runoff_mm_yr}, metadata, rainfall


def attach_model_unit_rainfall(units: gpd.GeoDataFrame, rainfall: pd.DataFrame, rainfall_defaults: dict) -> gpd.GeoDataFrame:
    if rainfall.empty or "raster_path" not in rainfall.columns:
        print("No rainfall rasters available for per-unit extraction; keeping AOI-wide rainfall defaults.")
        return units

    unit_ids = units["model_unit_id"].tolist()
    monthly_values = {unit_id: [] for unit_id in unit_ids}
    raster_fallback_dir = RAINFALL_SUMMARY_PATH.parent / "rasters"

    for _, row in rainfall.iterrows():
        raster_path = resolve_existing_path(row["raster_path"], raster_fallback_dir)
        if not raster_path.exists():
            print(f"Rainfall raster not found for per-unit extraction, skipping: {raster_path}")
            continue
        with rasterio.open(raster_path) as src:
            units_for_raster = units[["model_unit_id", "geometry"]].to_crs(src.crs)
            for unit_id, geom in zip(units_for_raster["model_unit_id"], units_for_raster.geometry):
                try:
                    clipped, _ = mask(src, [mapping(geom)], crop=True, nodata=src.nodata, filled=False)
                except ValueError:
                    monthly_values[unit_id].append(np.nan)
                    continue
                data = clipped[0]
                if src.nodata is not None:
                    data = np.ma.masked_equal(data, src.nodata)
                data = np.ma.masked_invalid(data)
                monthly_values[unit_id].append(float(data.mean()) if data.count() else np.nan)

    month_count = max((len(values) for values in monthly_values.values()), default=0)
    if month_count == 0:
        print("No per-unit rainfall values were extracted; keeping AOI-wide rainfall defaults.")
        return units

    units = units.copy()
    per_unit_precip = []
    for unit_id in unit_ids:
        values = pd.Series(monthly_values[unit_id], dtype="float64").dropna()
        if values.empty:
            per_unit_precip.append(rainfall_defaults["precipitation_mm_yr"])
        else:
            per_unit_precip.append(float(values.sum() * (12 / len(values))))
    units["precipitation_mm_yr"] = per_unit_precip
    units["runoff_mm_yr"] = units["precipitation_mm_yr"] * RUNOFF_FRACTION_OF_PRECIPITATION
    units["rainfall_unit_join"] = "mean_chirps_monthly_raster_by_cropland_model_unit_geometry"
    print(
        "Per-unit annualized precipitation range: "
        f"{units['precipitation_mm_yr'].min():.1f}-{units['precipitation_mm_yr'].max():.1f} mm/yr"
    )
    return units


soil_defaults = {
    "soil_ph": 6.2,
    "cec_cmolc_kg": 14.0,
    "clay_pct": 28.0,
    "bulk_density_g_cm3": 1.30,
    "soil_depth_cm": 30.0,
}
loaded_soil_defaults, soil_metadata = load_soil_defaults(SOIL_DEFAULTS_PATH)
soil_types = load_soil_types(SOIL_TYPES_PATH)
rainfall_defaults, rainfall_metadata, rainfall_summary = load_rainfall_defaults(RAINFALL_SUMMARY_PATH)
soil_defaults.update(loaded_soil_defaults)

# Start with AOI/default values, then overwrite soil and rainfall fields with
# per-model-unit values where the processed layers support it.
defaults = ScepterDefaults(
    **soil_defaults,
    temperature_c=24.0,
    precipitation_mm_yr=rainfall_defaults["precipitation_mm_yr"],
    runoff_mm_yr=rainfall_defaults["runoff_mm_yr"],
    basalt_application_t_ha=20.0,
    basalt_d50_um=50.0,
    simulation_years=10,
)

model_units = make_model_units(model_unit_blocks, defaults=defaults, max_units=None)
model_units["input_status"] = "hwsd2_soil_map_chirps_rainfall_runoff_estimate"

if INPUT_MODE == "esa_cropland_raster_blocks":
    model_units["block_geometry_area_m2"] = model_units.geometry.area
    model_units["area_m2"] = model_units["cropland_area_m2"]
    model_units["area_ha"] = model_units["cropland_area_ha"]
    model_units["input_status"] = "esa_cropland_pixel_geometry_with_hwsd2_soil_map_chirps_rainfall_runoff_estimate"

model_units = attach_soil_map_attributes(model_units, soil_types, SOIL_TYPES_PATH)
model_units = apply_per_unit_soil_inputs(model_units, soil_defaults)
model_units = attach_model_unit_rainfall(model_units, rainfall_summary, rainfall_defaults)

for key, value in {**soil_metadata, **rainfall_metadata}.items():
    model_units[key] = value

model_units["cropland_source"] = "ESA WorldCover cropland mask"

MODEL_UNIT_COLUMNS = [
    "model_unit_id",
    "area_ha",
    "centroid_lon",
    "centroid_lat",
    "cropland_source",
    "cropland_source_path",
    "cropland_pixels",
    "cropland_pixel_area_m2",
    "cropland_area_note",
    "cropland_geometry_note",
    "soil_map_hwsd2_unit_id",
    "soil_map_texture_group",
    "soil_map_wrb4",
    "soil_map_fao90",
    "soil_map_clay_pct",
    "soil_map_sand_pct",
    "soil_map_silt_pct",
    "soil_map_ph",
    "soil_map_cec_cmolc_kg",
    "soil_map_cec_clay_cmolc_kg",
    "soil_map_cec_eff_cmolc_kg",
    "soil_map_bulk_density_g_cm3",
    "soil_map_ref_bulk_density_g_cm3",
    "soil_map_source_path",
    "soil_map_join",
    "soil_ph",
    "cec_cmolc_kg",
    "clay_pct",
    "bulk_density_g_cm3",
    "soil_depth_cm",
    "temperature_c",
    "precipitation_mm_yr",
    "runoff_mm_yr",
    "rainfall_unit_join",
    "input_status",
    "soil_source",
    "soil_source_path",
    "soil_note",
    "soil_input_note",
    "rainfall_source",
    "rainfall_source_path",
    "rainfall_months_used",
    "missing_requested_months",
    "runoff_note",
]


def build_model_units_table(units: gpd.GeoDataFrame) -> pd.DataFrame:
    columns = [column for column in MODEL_UNIT_COLUMNS if column in units.columns]
    missing_required = [
        column
        for column in ["model_unit_id", "area_ha", "centroid_lon", "centroid_lat", "soil_map_texture_group", "soil_ph", "cec_cmolc_kg", "bulk_density_g_cm3", "precipitation_mm_yr"]
        if column not in columns
    ]
    if missing_required:
        raise ValueError(f"Model units are missing required real-data columns: {missing_required}")
    return pd.DataFrame(units[columns]).copy()


model_unit_table = build_model_units_table(model_units)

print(f"Model units prepared: {len(model_units):,}")
print(f"Units with soil map labels: {model_unit_table['soil_map_texture_group'].notna().sum():,}")
print(
    "SCEPTER soil pH range: "
    f"{model_unit_table['soil_ph'].min():.2f}-{model_unit_table['soil_ph'].max():.2f}"
)
print(
    "SCEPTER precipitation range: "
    f"{model_unit_table['precipitation_mm_yr'].min():.1f}-{model_unit_table['precipitation_mm_yr'].max():.1f} mm/yr"
)
model_unit_table.head()


Using HWSD2 soil defaults from: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv
Using CHIRPS rainfall summary from: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv
AOI annualized precipitation fallback: 1132.3 mm/yr
AOI estimated runoff fallback: 283.1 mm/yr


/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/src/erw_mrv/scepter.py:103: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  units["area_m2"] = units.geometry.area
/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/src/erw_mrv/scepter.py:106: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_projected = gpd.GeoSeries(units.geometry.centroid, crs=units.crs)
/var/folders/tk/wwspxgj910v5vdkljtjm383w0000gn/T/ipykernel_7977/1421909866.py:266: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  model_units["block_geometry_area_

Per-unit annualized precipitation range: 926.3-1200.2 mm/yr
Model units prepared: 100
Units with soil map labels: 100
SCEPTER soil pH range: 4.90-6.70
SCEPTER precipitation range: 926.3-1200.2 mm/yr


,model_unit_id,area_ha,centroid_lon,centroid_lat,cropland_source,cropland_source_path,cropland_pixels,cropland_pixel_area_m2,cropland_area_note,soil_map_hwsd2_unit_id,soil_map_texture_group,soil_map_wrb4,soil_map_fao90,soil_map_clay_pct,soil_map_sand_pct,soil_map_silt_pct,soil_map_ph,soil_map_cec_cmolc_kg,soil_map_cec_clay_cmolc_kg,soil_map_cec_eff_cmolc_kg,soil_map_bulk_density_g_cm3,soil_map_ref_bulk_density_g_cm3,soil_map_source_path,soil_map_join,soil_ph,cec_cmolc_kg,clay_pct,bulk_density_g_cm3,soil_depth_cm,temperature_c,precipitation_mm_yr,runoff_mm_yr,rainfall_unit_join,input_status,soil_source,soil_source_path,soil_note,soil_input_note,rainfall_source,rainfall_source_path,rainfall_months_used,missing_requested_months,runoff_note
0,mu_00001,1386.31,31.569750,1.085833,ESA WorldCover cropland mask,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_...,138631,100.0,geographic raster; used ESA WorldCover nominal 10 m pixel area,27462,Loam,FLdy,FLd,23,49,28,4.9,11,30,7,1.37,1.76,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_aoi_...,representative_point_within_hwsd2_aoi_soil_type,4.9,11.0,23.0,1.37,30.0,24.0,1051.341400,262.835350,mean_chirps_monthly_raster_by_model_unit_geometry,esa_cropland_block_with_hwsd2_soil_map_chirps_rainfall_runoff_estimate,erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_mdb_HWSD2_LAYERS.csv,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_so...,AOI pixel-share weighted HWSD2 defaults; per-model-unit SCEPTER inputs use representative-point HWSD2 attributes whe...,SCEPTER soil fields use per-model-unit HWSD2 representative-point attributes where available; otherwise AOI/default ...,CHIRPS rainfall_chirps_monthly via DE Africa S3,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/climate/rainfall/mont...,"2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12",nan,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr
1,mu_00002,1318.87,31.527083,0.659167,ESA WorldCover cropland mask,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_...,131887,100.0,geographic raster; used ESA WorldCover nominal 10 m pixel area,27470,Loam,FLdy,FLd,23,49,28,4.9,11,30,7,1.37,1.76,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_aoi_...,representative_point_within_hwsd2_aoi_soil_type,4.9,11.0,23.0,1.37,30.0,24.0,1123.615417,280.903854,mean_chirps_monthly_raster_by_model_unit_geometry,esa_cropland_block_with_hwsd2_soil_map_chirps_rainfall_runoff_estimate,erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_mdb_HWSD2_LAYERS.csv,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_so...,AOI pixel-share weighted HWSD2 defaults; per-model-unit SCEPTER inputs use representative-point HWSD2 attributes whe...,SCEPTER soil fields use per-model-unit HWSD2 representative-point attributes where available; otherwise AOI/default ...,CHIRPS rainfall_chirps_monthly via DE Africa S3,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/climate/rainfall/mont...,"2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12",nan,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr
2,mu_00003,1310.58,31.441750,1.171167,ESA WorldCover cropland mask,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_...,131058,100.0,geographic raster; used ESA WorldCover nominal 10 m pixel area,27462,Loam,FLdy,FLd,23,49,28,4.9,11,30,7,1.37,1.76,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_aoi_...,repre

## Define Scenarios

The scenarios table controls amendment rate, particle size, and simulation length. The baseline scenario is important because ERW results should be compared against no-amendment conditions.


In [5]:
scenarios = DEFAULT_SCENARIOS.copy()
scenarios


,scenario_id,basalt_application_t_ha,basalt_d50_um,simulation_years,description
0,baseline_no_erw,0.0,50.0,10,No rock amendment baseline.
1,erw_10t_fine,10.0,50.0,10,Low application rate with fine basalt.
2,erw_20t_fine,20.0,50.0,10,Middle application rate with fine basalt.
3,erw_40t_medium,40.0,100.0,10,Higher application rate with medium basalt.


## Expand To Run Table

This creates one row for every model unit and scenario. Later, each `run_id` can map to a SCEPTER configuration folder or input file.


In [6]:
runs = expand_scepter_runs(model_unit_table, scenarios)
print(f"SCEPTER runs staged: {len(runs):,}")
runs.head()


SCEPTER runs staged: 400


,run_id,model_unit_id,scenario_id,area_ha,centroid_lon,centroid_lat,cropland_source,cropland_source_path,cropland_pixels,cropland_pixel_area_m2,cropland_area_note,soil_map_hwsd2_unit_id,soil_map_texture_group,soil_map_wrb4,soil_map_fao90,soil_map_clay_pct,soil_map_sand_pct,soil_map_silt_pct,soil_map_ph,soil_map_cec_cmolc_kg,soil_map_cec_clay_cmolc_kg,soil_map_cec_eff_cmolc_kg,soil_map_bulk_density_g_cm3,soil_map_ref_bulk_density_g_cm3,soil_map_source_path,soil_map_join,soil_ph,cec_cmolc_kg,clay_pct,bulk_density_g_cm3,soil_depth_cm,temperature_c,precipitation_mm_yr,runoff_mm_yr,rainfall_unit_join,input_status,soil_source,soil_source_path,soil_note,soil_input_note,rainfall_source,rainfall_source_path,rainfall_months_used,missing_requested_months,runoff_note,basalt_application_t_ha,basalt_d50_um,simulation_years,description
0,mu_00001__baseline_no_erw,mu_00001,baseline_no_erw,1386.31,31.569750,1.085833,ESA WorldCover cropland mask,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_...,138631,100.0,geographic raster; used ESA WorldCover nominal 10 m pixel area,27462,Loam,FLdy,FLd,23,49,28,4.9,11,30,7,1.37,1.76,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_aoi_...,representative_point_within_hwsd2_aoi_soil_type,4.9,11.0,23.0,1.37,30.0,24.0,1051.341400,262.835350,mean_chirps_monthly_raster_by_model_unit_geometry,esa_cropland_block_with_hwsd2_soil_map_chirps_rainfall_runoff_estimate,erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_mdb_HWSD2_LAYERS.csv,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_so...,AOI pixel-share weighted HWSD2 defaults; per-model-unit SCEPTER inputs use representative-point HWSD2 attributes whe...,SCEPTER soil fields use per-model-unit HWSD2 representative-point attributes where available; otherwise AOI/default ...,CHIRPS rainfall_chirps_monthly via DE Africa S3,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/climate/rainfall/mont...,"2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12",nan,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr,0.0,50.0,10,No rock amendment baseline.
1,mu_00001__erw_10t_fine,mu_00001,erw_10t_fine,1386.31,31.569750,1.085833,ESA WorldCover cropland mask,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_...,138631,100.0,geographic raster; used ESA WorldCover nominal 10 m pixel area,27462,Loam,FLdy,FLd,23,49,28,4.9,11,30,7,1.37,1.76,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_aoi_...,representative_point_within_hwsd2_aoi_soil_type,4.9,11.0,23.0,1.37,30.0,24.0,1051.341400,262.835350,mean_chirps_monthly_raster_by_model_unit_geometry,esa_cropland_block_with_hwsd2_soil_map_chirps_rainfall_runoff_estimate,erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_mdb_HWSD2_LAYERS.csv,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_so...,AOI pixel-share weighted HWSD2 defaults; per-model-unit SCEPTER inputs use representative-point HWSD2 attributes whe...,SCEPTER soil fields use per-model-unit HWSD2 representative-point attributes where available; otherwise AOI/default ...,CHIRPS rainfall_chirps_monthly via DE Africa S3,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/climate/rainfall/mont...,"2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12",nan,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr,10.0,50.0,10,Low application rate with fine basalt.
2,mu_00001__erw_20t_fine,mu_00001,erw_20t_fine,1386.31,31.569750,1.085833,ESA WorldCover cropland mask,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/

## Write SCEPTER Staging Inputs


In [7]:
def write_real_scepter_inputs(
    units: gpd.GeoDataFrame,
    units_table: pd.DataFrame,
    runs_table: pd.DataFrame,
    scenarios_table: pd.DataFrame,
    output_dir: Path,
) -> dict[str, Path]:
    output_dir = ensure_dir(output_dir)

    units_gpkg_path = output_dir / "scepter_model_units.gpkg"
    units_table_path = output_dir / "scepter_model_units.csv"
    scenarios_path = output_dir / "scepter_scenarios.csv"
    runs_path = output_dir / "scepter_runs.csv"
    readme_path = output_dir / "README_scepter_inputs.md"

    units.to_file(units_gpkg_path, layer="model_units", driver="GPKG")
    units_table.to_csv(units_table_path, index=False)
    scenarios_table.to_csv(scenarios_path, index=False)
    runs_table.to_csv(runs_path, index=False)

    readme_text = """# SCEPTER Input Tables

These files are generated by notebook 04 using the prepared cropland map, HWSD2 soil map/defaults, and CHIRPS rainfall rasters.

- `scepter_model_units.gpkg`: spatial model units derived from ESA cropland raster blocks.
- `scepter_model_units.csv`: non-spatial model-unit attributes with cropland, per-unit HWSD2 soil attributes, per-unit CHIRPS rainfall, and runoff provenance.
- `scepter_scenarios.csv`: ERW application scenarios.
- `scepter_runs.csv`: one row per model unit and scenario, carrying the same input provenance for notebook 05.
"""
    readme_path.write_text(readme_text, encoding="utf-8")
    return {
        "model_units_gpkg": units_gpkg_path,
        "model_units_csv": units_table_path,
        "scenarios_csv": scenarios_path,
        "runs_csv": runs_path,
        "readme": readme_path,
    }


written = write_real_scepter_inputs(
    model_units,
    model_unit_table,
    runs,
    scenarios,
    SCEPTER_INPUT_DIR,
)

required_metadata_columns = [
    "cropland_source",
    "cropland_source_path",
    "soil_map_texture_group",
    "soil_map_source_path",
    "soil_source",
    "rainfall_source",
    "rainfall_months_used",
    "runoff_note",
]
missing_files = [label for label, path in written.items() if not Path(path).exists()]
if missing_files:
    raise FileNotFoundError(f"Notebook 04 did not write expected SCEPTER input files: {missing_files}")

written_runs = pd.read_csv(written["runs_csv"])
written_units = pd.read_csv(written["model_units_csv"])
missing_run_columns = [column for column in required_metadata_columns if column not in written_runs.columns]
missing_unit_columns = [column for column in required_metadata_columns if column not in written_units.columns]
if missing_run_columns or missing_unit_columns:
    raise ValueError(
        "Notebook 04 wrote SCEPTER files, but real input metadata columns are missing. "
        f"Missing from runs: {missing_run_columns}; missing from units: {missing_unit_columns}"
    )

missing_required_values = written_runs[required_metadata_columns].isna().sum()
missing_required_values = missing_required_values[missing_required_values > 0]
if not missing_required_values.empty:
    raise ValueError(
        "Notebook 04 wrote SCEPTER files, but required metadata values are still missing: "
        f"{missing_required_values.to_dict()}"
    )

print("Wrote SCEPTER input files:")
for label, path in written.items():
    print(f"- {label}: {path}")
print(f"Staged model units: {len(written_units):,}")
print(f"Staged SCEPTER runs: {len(written_runs):,}")
print("Cropland, soil-map, and rainfall metadata columns verified in scepter_runs.csv.")
written


Wrote SCEPTER input files:
- model_units_gpkg: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/scepter_model_units.gpkg
- model_units_csv: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/scepter_model_units.csv
- scenarios_csv: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/scepter_scenarios.csv
- runs_csv: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/scepter_runs.csv
- readme: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/README_scepter_inputs.md
Staged model units: 100
Staged SCEPTER runs: 400
Cropland, soil-map, and rainfall metadata columns verified in scepter_runs.csv.


{'model_units_gpkg': PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/scepter_model_units.gpkg'),
 'model_units_csv': PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/scepter_model_units.csv'),
 'scenarios_csv': PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/scepter_scenarios.csv'),
 'runs_csv': PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/scepter_runs.csv'),
 'readme': PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/inputs/README_scepter_inputs.md')}

## Quick QA

Check that every scenario has the same number of model units and that no required values are missing.


In [8]:
qa_by_scenario = runs.groupby("scenario_id").agg(
    run_count=("run_id", "count"),
    total_area_ha=("area_ha", "sum"),
    mean_application_t_ha=("basalt_application_t_ha", "mean"),
).reset_index()

required_columns = [
    "run_id",
    "model_unit_id",
    "scenario_id",
    "area_ha",
    "cropland_source_path",
    "soil_map_texture_group",
    "soil_ph",
    "temperature_c",
    "precipitation_mm_yr",
    "basalt_application_t_ha",
    "basalt_d50_um",
    "simulation_years",
]
missing_values = runs[required_columns].isna().sum().reset_index()
missing_values.columns = ["column", "missing_count"]

soil_texture_counts = (
    model_unit_table.groupby("soil_map_texture_group", dropna=False)
    .agg(model_units=("model_unit_id", "count"), cropland_area_ha=("area_ha", "sum"))
    .reset_index()
    .sort_values("cropland_area_ha", ascending=False)
)

qa_by_scenario, missing_values, soil_texture_counts


(       scenario_id  run_count  total_area_ha  mean_application_t_ha
 0  baseline_no_erw        100       90763.12                    0.0
 1     erw_10t_fine        100       90763.12                   10.0
 2     erw_20t_fine        100       90763.12                   20.0
 3   erw_40t_medium        100       90763.12                   40.0,
                      column  missing_count
 0                    run_id              0
 1             model_unit_id              0
 2               scenario_id              0
 3                   area_ha              0
 4      cropland_source_path              0
 5    soil_map_texture_group              0
 6                   soil_ph              0
 7             temperature_c              0
 8       precipitation_mm_yr              0
 9   basalt_application_t_ha              0
 10            basalt_d50_um              0
 11         simulation_years              0,
   soil_map_texture_group  model_units  cropland_area_ha
 2                   Loa

## Outputs From This Notebook

This notebook writes first-pass SCEPTER input staging files under `data/scepter_runs/inputs/`:

- `scepter_model_units.gpkg`: spatial cropland model units derived from ESA cropland raster blocks.
- `scepter_model_units.csv`: non-spatial model unit table with cropland area, centroid, soil-map labels, soil defaults, and climate attributes.
- `scepter_scenarios.csv`: baseline and ERW application scenarios.
- `scepter_runs.csv`: one row per model unit and scenario, with a unique `run_id`.
- `README_scepter_inputs.md`: notes explaining the staging files and current assumptions.

The current notebook is intentionally raster-first and does not compute vector field-level SCEPTER units.
